# 🤖 03: Bots Autónomos, Riqueza y Macroeconomía
Este notebook analiza la salud macroeconómica de Brokerdex, la velocidad del dinero y la distribución de riqueza entre los agentes.

**Preguntas clave a responder:**
1. ¿Cuál es la masa monetaria total ($M_1$) en circulación y en reservas?
2. ¿Cómo de equitativa o concentrada está la riqueza entre los 100 entrenadores (Curva de Lorenz y Coeficiente de Gini)?
3. ¿Cuánto dinero en comisiones recauda la plataforma?


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from db import get_engine
engine = get_engine()


## 1. Masa Monetaria del Sistema ($M_1$)
Calculamos la liquidez en saldos de usuarios excluyendo la cuenta interna del Market Maker.


In [ ]:
query_m1 = """
SELECT 
    COUNT(*) AS total_entrenadores,
    ROUND(SUM(balance_cents) / 100.0, 2) AS dinero_liquido_total_usd,
    ROUND(AVG(balance_cents) / 100.0, 2) AS saldo_promedio_usd,
    ROUND(MIN(balance_cents) / 100.0, 2) AS saldo_minimo_usd,
    ROUND(MAX(balance_cents) / 100.0, 2) AS saldo_maximo_usd
FROM account_account
WHERE is_active = true AND username != 'market_maker';
"""

df_m1 = pd.read_sql_query(query_m1, engine)
df_m1


## 2. Distribución de Riqueza y Coeficiente de Gini
Calculamos el patrimonio neto total de cada usuario (efectivo líquido + valor de mercado de sus criaturas).


In [ ]:
query_wealth = """
SELECT 
    a.username,
    ROUND(a.balance_cents / 100.0, 2) AS cash_usd,
    ROUND(COALESCE(portfolio_val.total_asset_value, 0), 2) AS asset_value_usd,
    ROUND((a.balance_cents / 100.0 + COALESCE(portfolio_val.total_asset_value, 0)), 2) AS net_worth_usd
FROM account_account a
LEFT JOIN (
    SELECT 
        p.owner_id,
        SUM(p.quantity * c.current_price) AS total_asset_value
    FROM trading_portfolio p
    JOIN creature_creature c ON p.creature_id = c.id
    GROUP BY p.owner_id
) portfolio_val ON a.id = portfolio_val.owner_id
WHERE a.is_active = true AND a.username != 'market_maker'
ORDER BY net_worth_usd ASC;
"""

df_wealth = pd.read_sql_query(query_wealth, engine)

# Cálculo de la Curva de Lorenz y Coeficiente de Gini
wealth_values = np.sort(df_wealth['net_worth_usd'].values)
n = len(wealth_values)
cum_wealth = np.cumsum(wealth_values)
lorenz_curve = cum_wealth / cum_wealth[-1]
equality_line = np.linspace(0, 1, n)

# Coeficiente de Gini
gini = (np.sum((2 * np.arange(1, n + 1) - n - 1) * wealth_values)) / (n * np.sum(wealth_values))

# Gráfico de Lorenz
fig_lorenz = go.Figure()
fig_lorenz.add_trace(go.Scatter(x=equality_line, y=equality_line, mode='lines', name='Igualdad Perfecta (Gini = 0)', line=dict(dash='dash', color='gray')))
fig_lorenz.add_trace(go.Scatter(x=equality_line, y=lorenz_curve, mode='lines', name=f'Curva de Lorenz Real (Gini = {gini:.3f})', line=dict(color='#00CC96', width=3)))

fig_lorenz.update_layout(
    title=f"Curva de Lorenz y Desigualdad de Riqueza (Coeficiente de Gini: {gini:.3f})",
    xaxis_title="Proporción Acumulada de Entrenadores",
    yaxis_title="Proporción Acumulada de Riqueza Total",
    template="plotly_dark",
    height=550
)
fig_lorenz.show()


## 3. Recaudación de Comisiones de la Plataforma
Total de ingresos generados por la comisión del 1.5% en cada fill ejecutado en el libro de órdenes.


In [ ]:
query_fees = """
SELECT 
    COUNT(*) AS total_trades,
    ROUND(SUM(commission), 2) AS comisiones_totales_usd,
    ROUND(AVG(commission), 2) AS comision_promedio_por_trade_usd,
    ROUND(SUM(quantity * price_per_unit), 2) AS volumen_total_usd
FROM trading_trade;
"""

df_fees = pd.read_sql_query(query_fees, engine)
df_fees
